# Assets · 02 Silver
Typed and checked. The **quality gate** runs every rule first; if one fails, nothing is written and silver keeps the previous version. Checks against other domains read their silver tables through shortcuts.

In [ ]:
# Domain: assets · owner: Service manager
# Attach this notebook to the lakehouse "lh_assets" in the workspace "Solvi Assets".
DOMAIN = "assets"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to f"{DOMAIN}_"
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named <domain>_<name>
    return spark.read.table(f"{domain}_{name}")

In [ ]:
SCHEMA = {
 "installation": {
  "installation_id": "string",
  "order_id": "string",
  "customer_id": "string",
  "site_id": "string",
  "installed_date": "date",
  "grid_connected_date": "date",
  "kwp": "double",
  "battery_kwh": "double",
  "orientation": "string",
  "tilt_deg": "int"
 },
 "production_daily": {
  "installation_id": "string",
  "date": "date",
  "kwh": "double"
 },
 "service_visit": {
  "visit_id": "string",
  "installation_id": "string",
  "visit_date": "date",
  "visit_type": "string",
  "hours": "double",
  "cost_nok": "decimal(12,2)"
 },
 "fault_log_truth": {
  "installation_id": "string",
  "fault_start": "date",
  "fault_end": "date",
  "fault_type": "string"
 }
}
KEYS = {"installation": ["installation_id"], "production_daily": ["installation_id", "date"], "service_visit": ["visit_id"], "fault_log_truth": ["installation_id", "fault_start"]}
FKS = [["installation", "order_id", "sales_order", "order_id", "sales"], ["installation", "customer_id", "customer", "customer_id", "core"], ["installation", "site_id", "site", "site_id", "core"], ["production_daily", "installation_id", "installation", "installation_id", "assets"], ["service_visit", "installation_id", "installation", "installation_id", "assets"]]      # child, column, parent, column, parent's domain
RANGES = [["production_daily", "kwh >= 0"], ["installation", "kwp > 0 AND kwp < 1000"]]

def clean(name):
    df = read_table(f"bronze_{name}")
    return df.select(*[F.nullif(F.trim(F.col(c)), F.lit("")).cast(t).alias(c) for c, t in SCHEMA[name].items()])
silver = {n: clean(n) for n in SCHEMA}
def parent(table, dom):
    return silver[table] if dom == DOMAIN else read_shared(dom, f"silver_{table}")

results = []
for t, keys in KEYS.items():
    df = silver[t]
    results += [(t, "key not null", df.filter(" OR ".join(f"{k} IS NULL" for k in keys)).count()),
                (t, "key unique", df.groupBy(*keys).count().filter("count > 1").count())]
for c, cc, p, pc, dom in FKS:
    orphans = silver[c].filter(F.col(cc).isNotNull()).join(parent(p, dom).select(F.col(pc).alias(cc)), cc, "left_anti").count()
    results.append((c, f"{cc} -> {dom}.{p}", orphans))
for t, cond in RANGES:
    results.append((t, cond, silver[t].filter(f"NOT ({cond})").count()))
dq = (spark.createDataFrame(results, "table string, rule string, failing_rows long")
        .withColumn("passed", F.col("failing_rows") == 0).withColumn("checked_at", F.current_timestamp()))
display(dq.orderBy("passed", "table"))
failed = dq.filter("NOT passed").count()
if failed:
    raise Exception(f"Quality gate failed in {DOMAIN}: {failed} rule(s). Nothing written to silver.")

In [ ]:
for name, df in silver.items():
    write_table(df, f"silver_{name}")
write_table(dq, "silver_dq_log")